In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuração e Verificação Inicial

In [2]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

MODEL_NAME = "dbmdz/bert-base-cased-finetuned-conll03-english"

In [3]:
def read_conll(path):
    tokens, tags = [], []
    sent_tokens, sent_tags = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()
            if not line:
                if tokens:
                    sent_tokens.append(tokens)
                    sent_tags.append(tags)
                    tokens, tags = [], []
            else:
                tok, _, _, ner = line.split()
                tokens.append(tok)
                tags.append(ner)
    # último sentença
    if tokens:
        sent_tokens.append(tokens)
        sent_tags.append(tags)
    return sent_tokens, sent_tags

In [4]:
def read_conll(path: Path, start_sentence_id: int = 0):
    """
    Lê arquivos CoNLL/ CleanCoNLL:
      • usa parts[0] como token
      • usa parts[-1] como rótulo NER (corrigido)
      • ignora linhas '-DOCSTART- …'
    """
    tokens, tags, sent_ids = [], [], []
    cur_toks, cur_tags = [], []
    sid = start_sentence_id

    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()

            if not line:  # fim da sentença
                if cur_toks:
                    tokens.append(cur_toks)
                    tags.append(cur_tags)
                    sent_ids.append(sid)
                    sid += 1
                    cur_toks, cur_tags = [], []
                continue

            parts = line.split()
            if parts[0] == "-DOCSTART-":  # pula marcador de doc
                continue

            tok, ner = parts[0], parts[-1]  # 1ª e última coluna
            cur_toks.append(tok)
            cur_tags.append(ner)

    if cur_toks:  # última sentença
        tokens.append(cur_toks)
        tags.append(cur_tags)
        sent_ids.append(sid)

    return tokens, tags, sent_ids, sid  # devolve sid para continuar contagem

In [5]:
def load_cleanconll(base_dir="~/Estudos/mestrado/data", keep_sentence_id=True) -> DatasetDict:
    base = Path(base_dir).expanduser()

    FILES = {  # filenames conforme seu print
        "train": "cleanconll.train",
        "dev": "cleanconll.dev",
        "test": "cleanconll.test",
    }

    splits, sid = {}, 0
    for split, fname in FILES.items():
        toks, labs, sids, sid = read_conll(base / fname, sid)
        data = {"tokens": toks, "ner_tags": labs}
        if keep_sentence_id:
            data["sentence_id"] = sids
        splits[split] = Dataset.from_dict(data)

    return DatasetDict(splits)

In [6]:
cleanconll_ds = load_cleanconll()  # pronto!
print(cleanconll_ds)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'sentence_id'],
        num_rows: 13957
    })
    dev: Dataset({
        features: ['tokens', 'ner_tags', 'sentence_id'],
        num_rows: 3233
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'sentence_id'],
        num_rows: 3427
    })
})


In [7]:
cleanconll_ds["train"] = cleanconll_ds["train"].add_column(
    "split", ["train"] * len(cleanconll_ds["train"])
)
cleanconll_ds["dev"] = cleanconll_ds["dev"].add_column(
    "split", ["dev"] * len(cleanconll_ds["dev"])
)
cleanconll_ds["test"] = cleanconll_ds["test"].add_column(
    "split", ["test"] * len(cleanconll_ds["test"])
)

cleanconll_full = concatenate_datasets(
    [
        cleanconll_ds["train"],
        cleanconll_ds["dev"],
        cleanconll_ds["test"],
    ]
)

In [8]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in cleanconll_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [9]:
id2label

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'B-PER',
 4: 'I-LOC',
 5: 'I-MISC',
 6: 'I-ORG',
 7: 'I-PER',
 8: 'O'}

In [10]:
NUM_LABELS

9

# Splits

In [11]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [12]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [13]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [14]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    print(f"Selecionando {int(pct_test*len(dataset))} sentenças para teste...")
    while len(test_idx) < int(pct_test*len(dataset)):
        print(f"  {len(test_idx)} selecionadas...")
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [15]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [16]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [17]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def standard_split_conll(dataset):
    return cleanconll_ds

In [19]:
# standard_split = standard_split_conll(cleanconll_full)
# print('std')
# # random_splt = random_splits(cleanconll_full)
# # print('random')
# heur_len = heur_len_split(cleanconll_full)
# print("heur_len")
# heur_rare = heur_rare_split(cleanconll_full)
# print("heur_rare")
# advers = adversarial_split(cleanconll_full)
# print("advs")
# loc = loc_split(cleanconll_full)
# print("loc")
# semantic = semantic_cluster_split(cleanconll_full)
# print("semantic")
# reverse = reverse_curriculum_split(cleanconll_full)
# print("reverse")

In [20]:
gc.collect()  # força o GC do Python
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

# Experimentos

In [21]:
from sklearn.metrics import f1_score as skl_f1

In [22]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc": loc_split,
            "reverse": reverse_curriculum_split,
            "semantic": semantic_cluster_split,
            "heur_len": heur_len_split,
            "heur_rare": heur_rare_split,
            "std": standard_split_conll,
            "advs": adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []  # p/ seqeval
        flat_preds, flat_labels = [], []  # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro = skl_f1(flat_labels, flat_preds, average="micro", zero_division=0)
        f1_macro = skl_f1(flat_labels, flat_preds, average="macro", zero_division=0)
        f1_weighted = skl_f1(
            flat_labels, flat_preds, average="weighted", zero_division=0
        )

        return {
            **seqeval_metrics,  # overall_precision / recall / f1
            "f1_micro": f1_micro,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }

    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        # output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="no",
        # load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none",
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [23]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "advs"]

In [24]:
results = {}
trainer_all = {}
s = splits[0]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: loc


/tmp/ipykernel_33487/4053031328.py:28: RuntimeWarning: invalid value encountered in divide
  topk[i] = (inter / uni).mean()
Map: 100%|██████████| 4123/4123 [00:00<00:00, 15950.85 examples/s]
/tmp/ipykernel_33487/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.165700,0.050227,"{'precision': 0.9418367346938775, 'recall': 0.918407960199005, 'f1': 0.9299748110831234, 'number': 1005}","{'precision': 0.8710865561694291, 'recall': 0.8269230769230769, 'f1': 0.8484304932735426, 'number': 572}","{'precision': 0.8369462770970783, 'recall': 0.9240374609781478, 'f1': 0.8783382789317506, 'number': 961}","{'precision': 0.974694589877836, 'recall': 0.9563356164383562, 'f1': 0.9654278305963699, 'number': 1168}",0.911796,0.917701,0.914739,0.985338,0.985338,0.915034,0.985348
2,0.016400,0.057198,"{'precision': 0.9475806451612904, 'recall': 0.9353233830845771, 'f1': 0.9414121181772659, 'number': 1005}","{'precision': 0.9105367793240556, 'recall': 0.8006993006993007, 'f1': 0.8520930232558139, 'number': 572}","{'precision': 0.8222836095764272, 'recall': 0.9292403746097815, 'f1': 0.872496336101612, 'number': 961}","{'precision': 0.9799126637554585, 'recall': 0.9606164383561644, 'f1': 0.9701686121919584, 'number': 1168}",0.915996,0.920939,0.918461,0.986089,0.986089,0.917853,0.986049
3,0.007000,0.052550,"{'precision': 0.9433962264150944, 'recall': 0.945273631840796, 'f1': 0.9443339960238568, 'number': 1005}","{'precision': 0.8851851851851852, 'recall': 0.8356643356643356, 'f1': 0.8597122302158274, 'number': 572}","{'precision': 0.8940936863543788, 'recall': 0.9136316337148803, 'f1': 0.9037570766855378, 'number': 961}","{'precision': 0.9725792630676949, 'recall': 0.9717465753424658, 'f1': 0.9721627408993576, 'number': 1168}",0.931006,0.928494,0.929749,0.987852,0.987852,0.929625,0.987781
4,0.002600,0.063565,"{'precision': 0.9431137724550899, 'recall': 0.9402985074626866, 'f1': 0.9417040358744396, 'number': 1005}","{'precision': 0.8761384335154827, 'recall': 0.8409090909090909, 'f1': 0.8581623550401428, 'number': 572}","{'precision': 0.8853439680957128, 'recall': 0.9240374609781478, 'f1': 0.9042769857433809, 'number': 961}","{'precision': 0.9667235494880546, 'recall': 0.9700342465753424, 'f1': 0.9683760683760683, 'number': 1168}",0.925121,0.930113,0.927610,0.987558,0.987558,0.927811,0.987558
5,0.001200,0.062212,"{'precision': 0.9493041749502982, 'recall': 0.9502487562189055, 'f1': 0.9497762307309796, 'number': 1005}","{'precision': 0.8656195462478184, 'recall': 0.8671328671328671, 'f1': 0.8663755458515283, 'number': 572}","{'precision': 0.9026639344262295, 'recall': 0.9167533818938606, 'f1': 0.9096541042849767, 'number': 961}","{'precision': 0.9742268041237113, 'recall': 0.9708904109589042, 'f1': 0.9725557461406518, 'number': 1168}",0.931971,0.935240,0.933603,0.988407,0.988407,0.930589,0.988428


F1 Macro: 0.93500454349467
F1 Micro: 0.9911462143720979
F1 Weighted: 0.9911140518742918
{'eval_loss': 0.05453643947839737, 'eval_LOC': {'precision': 0.9746906636670416, 'recall': 0.9569298729983434, 'f1': 0.9657286152131512, 'number': 1811}, 'eval_MISC': {'precision': 0.8830297219558965, 'recall': 0.8664158043273753, 'f1': 0.8746438746438746, 'number': 1063}, 'eval_ORG': {'precision': 0.9037520391517129, 'recall': 0.9400452488687783, 'f1': 0.9215414471860272, 'number': 1768}, 'eval_PER': {'precision': 0.9826043737574552, 'recall': 0.9782285997031173, 'f1': 0.9804116042648153, 'number': 2021}, 'eval_overall_precision': 0.9431954436450839, 'eval_overall_recall': 0.9444694582020111, 'eval_overall_f1': 0.9438320209973753, 'eval_overall_accuracy': 0.9911462143720979, 'eval_f1_micro': 0.9911462143720979, 'eval_f1_macro': 0.93500454349467, 'eval_f1_weighted': 0.9911140518742918, 'eval_runtime': 13.1932, 'eval_samples_per_second': 312.509, 'eval_steps_per_second': 19.556, 'epoch': 5.0}




20

In [25]:
!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
import time

In [27]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [28]:
results = {}
trainer_all = {}
s = splits[1]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: reverse


Map: 100%|██████████| 4123/4123 [00:00<00:00, 13053.84 examples/s]
/tmp/ipykernel_33487/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.151700,0.052137,"{'precision': 0.9853090172239108, 'recall': 0.9131455399061033, 'f1': 0.9478557504873293, 'number': 2130}","{'precision': 0.8329718004338394, 'recall': 0.8152866242038217, 'f1': 0.8240343347639484, 'number': 471}","{'precision': 0.7991543340380549, 'recall': 0.9310344827586207, 'f1': 0.8600682593856654, 'number': 812}","{'precision': 0.9735537190082645, 'recall': 0.990748528174937, 'f1': 0.9820758649437266, 'number': 1189}",0.928556,0.926336,0.927445,0.986793,0.986793,0.926414,0.986832
2,0.012800,0.044237,"{'precision': 0.982532751091703, 'recall': 0.9507042253521126, 'f1': 0.9663564781675018, 'number': 2130}","{'precision': 0.9073634204275535, 'recall': 0.8110403397027601, 'f1': 0.8565022421524663, 'number': 471}","{'precision': 0.8424507658643327, 'recall': 0.9482758620689655, 'f1': 0.8922363847045192, 'number': 812}","{'precision': 0.9816360601001669, 'recall': 0.9890664423885618, 'f1': 0.9853372434017595, 'number': 1189}",0.947540,0.945893,0.946716,0.989695,0.989695,0.939506,0.989614
3,0.005600,0.040645,"{'precision': 0.9778301886792453, 'recall': 0.9732394366197183, 'f1': 0.975529411764706, 'number': 2130}","{'precision': 0.8640350877192983, 'recall': 0.8365180467091295, 'f1': 0.8500539374325783, 'number': 471}","{'precision': 0.9063981042654028, 'recall': 0.9421182266009852, 'f1': 0.9239130434782609, 'number': 812}","{'precision': 0.9816360601001669, 'recall': 0.9890664423885618, 'f1': 0.9853372434017595, 'number': 1189}",0.954526,0.957844,0.956182,0.991827,0.991827,0.947126,0.991766
4,0.001800,0.045832,"{'precision': 0.9795432921027593, 'recall': 0.9666666666666667, 'f1': 0.973062381852552, 'number': 2130}","{'precision': 0.8876404494382022, 'recall': 0.8386411889596603, 'f1': 0.8624454148471615, 'number': 471}","{'precision': 0.8844393592677345, 'recall': 0.9519704433497537, 'f1': 0.9169632265717675, 'number': 812}","{'precision': 0.9849624060150376, 'recall': 0.9915895710681245, 'f1': 0.9882648784576697, 'number': 1189}",0.954093,0.957410,0.955748,0.991738,0.991738,0.946124,0.991667
5,0.000800,0.047185,"{'precision': 0.98286530223703, 'recall': 0.9694835680751174, 'f1': 0.9761285748050106, 'number': 2130}","{'precision': 0.9323671497584541, 'recall': 0.8195329087048833, 'f1': 0.872316384180791, 'number': 471}","{'precision': 0.8836158192090395, 'recall': 0.9630541871921182, 'f1': 0.9216263995285798, 'number': 812}","{'precision': 0.9841534612176814, 'recall': 0.992430613961312, 'f1': 0.9882747068676717, 'number': 1189}",0.959556,0.958931,0.959244,0.991946,0.991946,0.948899,0.991851


F1 Macro: 0.8924566318176258
F1 Micro: 0.9738698777145499
F1 Weighted: 0.972606856269624
{'eval_loss': 0.19054260849952698, 'eval_LOC': {'precision': 0.8916500994035785, 'recall': 0.9112766677954622, 'f1': 0.9013565566906716, 'number': 2953}, 'eval_MISC': {'precision': 0.9324546952224053, 'recall': 0.7239703248912766, 'f1': 0.8150921658986175, 'number': 3909}, 'eval_ORG': {'precision': 0.7406896551724138, 'recall': 0.9065841305571187, 'f1': 0.8152834008097166, 'number': 1777}, 'eval_PER': {'precision': 0.9539810080350621, 'recall': 0.9871504157218443, 'f1': 0.9702823179791976, 'number': 2646}, 'eval_overall_precision': 0.8885646543862848, 'eval_overall_recall': 0.8634470536109881, 'eval_overall_f1': 0.8758258055817716, 'eval_overall_accuracy': 0.9738698777145499, 'eval_f1_micro': 0.9738698777145499, 'eval_f1_macro': 0.8924566318176258, 'eval_f1_weighted': 0.972606856269624, 'eval_runtime': 17.826, 'eval_samples_per_second': 231.291, 'eval_steps_per_second': 14.473, 'epoch': 5.0}




0

In [29]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [30]:
results = {}
trainer_all = {}
s = splits[2]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: semantic


Map: 100%|██████████| 4180/4180 [00:00<00:00, 29326.14 examples/s]
/tmp/ipykernel_33487/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.163300,0.052087,"{'precision': 0.9470655926352128, 'recall': 0.9569767441860465, 'f1': 0.9519953730480045, 'number': 860}","{'precision': 0.8787346221441125, 'recall': 0.831946755407654, 'f1': 0.8547008547008547, 'number': 601}","{'precision': 0.9395744680851064, 'recall': 0.9443969204448246, 'f1': 0.9419795221843004, 'number': 1169}","{'precision': 0.9578488372093024, 'recall': 0.9894894894894894, 'f1': 0.9734121122599704, 'number': 666}",0.934868,0.936286,0.935577,0.987365,0.987365,0.935719,0.987174
2,0.017100,0.051333,"{'precision': 0.9536500579374276, 'recall': 0.9569767441860465, 'f1': 0.955310504933256, 'number': 860}","{'precision': 0.8916666666666667, 'recall': 0.8901830282861897, 'f1': 0.890924229808493, 'number': 601}","{'precision': 0.9556313993174061, 'recall': 0.9580838323353293, 'f1': 0.9568560444254592, 'number': 1169}","{'precision': 0.9791044776119403, 'recall': 0.984984984984985, 'f1': 0.9820359281437127, 'number': 666}",0.948260,0.950850,0.949553,0.989810,0.989810,0.949124,0.989755
3,0.007400,0.058025,"{'precision': 0.9625730994152046, 'recall': 0.9569767441860465, 'f1': 0.9597667638483965, 'number': 860}","{'precision': 0.8794788273615635, 'recall': 0.8985024958402662, 'f1': 0.8888888888888888, 'number': 601}","{'precision': 0.9474576271186441, 'recall': 0.9563729683490163, 'f1': 0.951894423158791, 'number': 1169}","{'precision': 0.9547445255474453, 'recall': 0.9819819819819819, 'f1': 0.9681717246484086, 'number': 666}",0.940312,0.951153,0.945701,0.989036,0.989036,0.944917,0.989006
4,0.002700,0.057833,"{'precision': 0.9548611111111112, 'recall': 0.9593023255813954, 'f1': 0.9570765661252901, 'number': 860}","{'precision': 0.9182608695652174, 'recall': 0.8785357737104825, 'f1': 0.8979591836734694, 'number': 601}","{'precision': 0.9558198810535259, 'recall': 0.962360992301112, 'f1': 0.9590792838874681, 'number': 1169}","{'precision': 0.9702380952380952, 'recall': 0.978978978978979, 'f1': 0.9745889387144993, 'number': 666}",0.951946,0.949636,0.950790,0.989892,0.989892,0.949148,0.989824
5,0.001700,0.057534,"{'precision': 0.9614035087719298, 'recall': 0.9558139534883721, 'f1': 0.9586005830903789, 'number': 860}","{'precision': 0.9074074074074074, 'recall': 0.8968386023294509, 'f1': 0.902092050209205, 'number': 601}","{'precision': 0.9608843537414966, 'recall': 0.9666381522668948, 'f1': 0.9637526652452025, 'number': 1169}","{'precision': 0.9717682020802377, 'recall': 0.9819819819819819, 'f1': 0.9768483943241225, 'number': 666}",0.953608,0.954187,0.953897,0.990259,0.990259,0.949353,0.990231


F1 Macro: 0.9499103603531791
F1 Micro: 0.986037334952193
F1 Weighted: 0.9854992335024628
{'eval_loss': 0.11199788749217987, 'eval_LOC': {'precision': 0.9740034662045061, 'recall': 0.9802325581395349, 'f1': 0.9771080846131557, 'number': 1720}, 'eval_MISC': {'precision': 0.7997118155619597, 'recall': 0.7676348547717843, 'f1': 0.7833450952717008, 'number': 723}, 'eval_ORG': {'precision': 0.9821063394683026, 'recall': 0.9761178861788617, 'f1': 0.9791029561671762, 'number': 1968}, 'eval_PER': {'precision': 0.9660493827160493, 'recall': 0.9750778816199377, 'f1': 0.9705426356589147, 'number': 321}, 'eval_overall_precision': 0.9511158342189161, 'eval_overall_recall': 0.9456889264581573, 'eval_overall_f1': 0.9483946169333476, 'eval_overall_accuracy': 0.986037334952193, 'eval_f1_micro': 0.986037334952193, 'eval_f1_macro': 0.9499103603531791, 'eval_f1_weighted': 0.9854992335024628, 'eval_runtime': 7.5487, 'eval_samples_per_second': 553.735, 'eval_steps_per_second': 34.708, 'epoch': 5.0}




0

In [31]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [32]:
results = {}
trainer_all = {}
s = splits[3]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_len


Map: 100%|██████████| 4123/4123 [00:00<00:00, 11432.19 examples/s]
/tmp/ipykernel_33487/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.170600,0.022312,"{'precision': 0.9735632183908046, 'recall': 0.968, 'f1': 0.9707736389684815, 'number': 875}","{'precision': 0.9024390243902439, 'recall': 0.9144486692015209, 'f1': 0.9084041548630785, 'number': 526}","{'precision': 0.9440879926672777, 'recall': 0.9563602599814299, 'f1': 0.9501845018450186, 'number': 1077}","{'precision': 0.9764373232799246, 'recall': 0.9923371647509579, 'f1': 0.9843230403800475, 'number': 1044}",0.954712,0.963657,0.959163,0.993644,0.993644,0.966613,0.993651
2,0.017600,0.020998,"{'precision': 0.9882629107981221, 'recall': 0.9622857142857143, 'f1': 0.9751013317892299, 'number': 875}","{'precision': 0.9097605893186004, 'recall': 0.9391634980988594, 'f1': 0.9242282507015903, 'number': 526}","{'precision': 0.9500454132606722, 'recall': 0.9712163416898792, 'f1': 0.960514233241506, 'number': 1077}","{'precision': 0.9885167464114832, 'recall': 0.9894636015325671, 'f1': 0.9889899473432264, 'number': 1044}",0.964417,0.969620,0.967011,0.994922,0.994922,0.971695,0.994937
3,0.008900,0.020555,"{'precision': 0.9706546275395034, 'recall': 0.9828571428571429, 'f1': 0.97671777399205, 'number': 875}","{'precision': 0.9239332096474954, 'recall': 0.9467680608365019, 'f1': 0.9352112676056338, 'number': 526}","{'precision': 0.9665738161559888, 'recall': 0.9665738161559888, 'f1': 0.9665738161559888, 'number': 1077}","{'precision': 0.9951783992285439, 'recall': 0.9885057471264368, 'f1': 0.9918308505526189, 'number': 1044}",0.969483,0.974162,0.971817,0.995426,0.995426,0.975729,0.995433
4,0.002300,0.023347,"{'precision': 0.9872241579558653, 'recall': 0.9714285714285714, 'f1': 0.9792626728110599, 'number': 875}","{'precision': 0.941398865784499, 'recall': 0.9467680608365019, 'f1': 0.9440758293838863, 'number': 526}","{'precision': 0.9595959595959596, 'recall': 0.9702878365831012, 'f1': 0.9649122807017545, 'number': 1077}","{'precision': 0.9857142857142858, 'recall': 0.9913793103448276, 'f1': 0.9885386819484241, 'number': 1044}",0.971380,0.973311,0.972344,0.995527,0.995527,0.977995,0.995523
5,0.001400,0.022361,"{'precision': 0.9805045871559633, 'recall': 0.9771428571428571, 'f1': 0.9788208357183743, 'number': 875}","{'precision': 0.941398865784499, 'recall': 0.9467680608365019, 'f1': 0.9440758293838863, 'number': 526}","{'precision': 0.966789667896679, 'recall': 0.9730733519034355, 'f1': 0.9699213327163351, 'number': 1077}","{'precision': 0.9904397705544933, 'recall': 0.9923371647509579, 'f1': 0.9913875598086125, 'number': 1044}",0.973379,0.975866,0.974621,0.995763,0.995763,0.977631,0.995764


F1 Macro: 0.9626374281964281
F1 Micro: 0.9929633431570581
F1 Weighted: 0.9929413692895563
{'eval_loss': 0.03745320439338684, 'eval_LOC': {'precision': 0.965897166841553, 'recall': 0.9745897300158815, 'f1': 0.9702239789196311, 'number': 1889}, 'eval_MISC': {'precision': 0.902135231316726, 'recall': 0.9151624548736462, 'f1': 0.9086021505376344, 'number': 1108}, 'eval_ORG': {'precision': 0.9567762269248087, 'recall': 0.9520609318996416, 'f1': 0.954412755445767, 'number': 2232}, 'eval_PER': {'precision': 0.9884653961885657, 'recall': 0.9874749498997996, 'f1': 0.98796992481203, 'number': 1996}, 'eval_overall_precision': 0.9594202898550724, 'eval_overall_recall': 0.962076124567474, 'eval_overall_f1': 0.9607463718037319, 'eval_overall_accuracy': 0.9929633431570581, 'eval_f1_micro': 0.9929633431570581, 'eval_f1_macro': 0.9626374281964281, 'eval_f1_weighted': 0.9929413692895563, 'eval_runtime': 13.7533, 'eval_samples_per_second': 299.782, 'eval_steps_per_second': 18.759, 'epoch': 5.0}




0

In [33]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [34]:
results = {}
trainer_all = {}
s = splits[4]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_rare


Map: 100%|██████████| 4123/4123 [00:00<00:00, 14928.02 examples/s]
/tmp/ipykernel_33487/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.169200,0.027930,"{'precision': 0.9760348583877996, 'recall': 0.9542066027689031, 'f1': 0.9649973074851912, 'number': 939}","{'precision': 0.8866799204771372, 'recall': 0.8814229249011858, 'f1': 0.8840436075322101, 'number': 506}","{'precision': 0.9527207850133809, 'recall': 0.9744525547445255, 'f1': 0.9634641407307172, 'number': 1096}","{'precision': 0.9820627802690582, 'recall': 0.9875986471251409, 'f1': 0.984822934232715, 'number': 887}",0.956902,0.958576,0.957738,0.992538,0.992538,0.958711,0.992495
2,0.015000,0.031272,"{'precision': 0.9621052631578947, 'recall': 0.9733759318423855, 'f1': 0.9677077818951826, 'number': 939}","{'precision': 0.8967495219885278, 'recall': 0.9268774703557312, 'f1': 0.91156462585034, 'number': 506}","{'precision': 0.9696691176470589, 'recall': 0.9625912408759124, 'f1': 0.9661172161172162, 'number': 1096}","{'precision': 0.9898305084745763, 'recall': 0.9875986471251409, 'f1': 0.9887133182844244, 'number': 887}",0.961695,0.966744,0.964213,0.993625,0.993625,0.966155,0.993649
3,0.006500,0.031253,"{'precision': 0.9682875264270613, 'recall': 0.9755058572949947, 'f1': 0.9718832891246685, 'number': 939}","{'precision': 0.9173228346456693, 'recall': 0.9209486166007905, 'f1': 0.9191321499013807, 'number': 506}","{'precision': 0.974145891043398, 'recall': 0.9625912408759124, 'f1': 0.9683340982101881, 'number': 1096}","{'precision': 0.9821029082774049, 'recall': 0.9898534385569335, 'f1': 0.9859629421673217, 'number': 887}",0.966191,0.967036,0.966613,0.993842,0.993842,0.966902,0.993835
4,0.003200,0.029875,"{'precision': 0.9795479009687836, 'recall': 0.9691160809371672, 'f1': 0.9743040685224839, 'number': 939}","{'precision': 0.9285714285714286, 'recall': 0.924901185770751, 'f1': 0.9267326732673268, 'number': 506}","{'precision': 0.9620938628158845, 'recall': 0.9726277372262774, 'f1': 0.9673321234119783, 'number': 1096}","{'precision': 0.9865319865319865, 'recall': 0.9909808342728298, 'f1': 0.9887514060742407, 'number': 887}",0.968240,0.969370,0.968805,0.994349,0.994349,0.969896,0.994330
5,0.001600,0.030924,"{'precision': 0.9722814498933902, 'recall': 0.9712460063897763, 'f1': 0.971763452317528, 'number': 939}","{'precision': 0.9261477045908184, 'recall': 0.9169960474308301, 'f1': 0.9215491559086395, 'number': 506}","{'precision': 0.9689213893967094, 'recall': 0.9671532846715328, 'f1': 0.9680365296803652, 'number': 1096}","{'precision': 0.9854423292273237, 'recall': 0.992108229988726, 'f1': 0.9887640449438202, 'number': 887}",0.967893,0.967328,0.967610,0.994205,0.994205,0.968110,0.994174


F1 Macro: 0.9449677526276847
F1 Micro: 0.9908256880733946
F1 Weighted: 0.9907451059492537
{'eval_loss': 0.05230069160461426, 'eval_LOC': {'precision': 0.9626966292134832, 'recall': 0.9588182632050134, 'f1': 0.9607535321821037, 'number': 2234}, 'eval_MISC': {'precision': 0.9022191400832178, 'recall': 0.8880546075085324, 'f1': 0.8950808393532852, 'number': 1465}, 'eval_ORG': {'precision': 0.9093484419263456, 'recall': 0.9277456647398844, 'f1': 0.9184549356223175, 'number': 2076}, 'eval_PER': {'precision': 0.9765355417529331, 'recall': 0.9853760445682451, 'f1': 0.9809358752166378, 'number': 2872}, 'eval_overall_precision': 0.9442588966946908, 'eval_overall_recall': 0.9481901237423384, 'eval_overall_f1': 0.9462204270051933, 'eval_overall_accuracy': 0.9908256880733946, 'eval_f1_micro': 0.9908256880733946, 'eval_f1_macro': 0.9449677526276847, 'eval_f1_weighted': 0.9907451059492537, 'eval_runtime': 16.1766, 'eval_samples_per_second': 254.875, 'eval_steps_per_second': 15.949, 'epoch': 5.0}




0

In [35]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
torch.cuda.ipc_collect()  # limpa buffers de IPC
torch.cuda.reset_accumulated_memory_stats()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[5]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(cleanconll_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: advs
Selecionando 4123 sentenças para teste…
  0 selecionadas…
  21 selecionadas…
  41 selecionadas…
  65 selecionadas…
  84 selecionadas…
  107 selecionadas…
  127 selecionadas…
  152 selecionadas…
  175 selecionadas…
  194 selecionadas…
  215 selecionadas…
  234 selecionadas…
  259 selecionadas…
  284 selecionadas…
  307 selecionadas…
  325 selecionadas…
  343 selecionadas…
  368 selecionadas…
  394 selecionadas…
  415 selecionadas…
  436 selecionadas…
  455 selecionadas…
  477 selecionadas…
  498 selecionadas…
  518 selecionadas…
  540 selecionadas…
  558 selecionadas…
  576 selecionadas…
  589 selecionadas…
  605 selecionadas…
  624 selecionadas…
  646 selecionadas…
  671 selecionadas…
  690 selecionadas…
  706 selecionadas…
  731 selecionadas…
  745 selecionadas…
  769 selecionadas…
  787 selecionadas…
  804 selecionadas…
  825 selecionadas…
  846 selecionadas…
  868 selecionadas…
  880 selecionadas…
  901 selecionadas…
  925 selecionadas…
  943 selecionadas…


Map: 100%|██████████| 4123/4123 [00:00<00:00, 7801.82 examples/s]
/tmp/ipykernel_33487/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Loc,Misc,Org,Per,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,0.171300,0.032508,"{'precision': 0.9556412729026037, 'recall': 0.970617042115573, 'f1': 0.9630709426627794, 'number': 1021}","{'precision': 0.9473684210526315, 'recall': 0.8615384615384616, 'f1': 0.9024171888988362, 'number': 585}","{'precision': 0.9151785714285714, 'recall': 0.950834879406308, 'f1': 0.932666060054595, 'number': 1078}","{'precision': 0.9774436090225563, 'recall': 0.985781990521327, 'f1': 0.9815950920245398, 'number': 1055}",0.948574,0.952126,0.950347,0.991485,0.991485,0.955024,0.991410
2,0.017200,0.030428,"{'precision': 0.9678988326848249, 'recall': 0.9745347698334965, 'f1': 0.9712054660810151, 'number': 1021}","{'precision': 0.9098497495826378, 'recall': 0.9316239316239316, 'f1': 0.920608108108108, 'number': 585}","{'precision': 0.9559925093632958, 'recall': 0.9471243042671614, 'f1': 0.9515377446411929, 'number': 1078}","{'precision': 0.9801512287334594, 'recall': 0.9829383886255925, 'f1': 0.9815428300993847, 'number': 1055}",0.958700,0.962289,0.960491,0.992746,0.992746,0.960571,0.992733
3,0.007900,0.033629,"{'precision': 0.9763079960513327, 'recall': 0.9686581782566112, 'f1': 0.9724680432645034, 'number': 1021}","{'precision': 0.913477537437604, 'recall': 0.9384615384615385, 'f1': 0.9258010118043846, 'number': 585}","{'precision': 0.9372197309417041, 'recall': 0.9693877551020408, 'f1': 0.953032375740994, 'number': 1078}","{'precision': 0.9866793529971456, 'recall': 0.9829383886255925, 'f1': 0.9848053181386515, 'number': 1055}",0.957672,0.968173,0.962894,0.993251,0.993251,0.964363,0.993235
4,0.003300,0.032997,"{'precision': 0.976401179941003, 'recall': 0.9725759059745348, 'f1': 0.9744847890088322, 'number': 1021}","{'precision': 0.927487352445194, 'recall': 0.9401709401709402, 'f1': 0.933786078098472, 'number': 585}","{'precision': 0.954337899543379, 'recall': 0.9693877551020408, 'f1': 0.9618039576622182, 'number': 1078}","{'precision': 0.9839622641509433, 'recall': 0.9886255924170616, 'f1': 0.9862884160756502, 'number': 1055}",0.964409,0.971115,0.967751,0.993787,0.993787,0.966720,0.993777
5,0.001600,0.035340,"{'precision': 0.9773399014778326, 'recall': 0.9715964740450539, 'f1': 0.9744597249508842, 'number': 1021}","{'precision': 0.9305084745762712, 'recall': 0.9384615384615385, 'f1': 0.934468085106383, 'number': 585}","{'precision': 0.9489981785063752, 'recall': 0.9666048237476809, 'f1': 0.9577205882352942, 'number': 1078}","{'precision': 0.9839622641509433, 'recall': 0.9886255924170616, 'f1': 0.9862884160756502, 'number': 1055}",0.963593,0.969778,0.966676,0.993661,0.993661,0.965364,0.993650


F1 Macro: 0.9473213166976306
F1 Micro: 0.9903980413534553
F1 Weighted: 0.9903744973929919
{'eval_loss': 0.05383947491645813, 'eval_LOC': {'precision': 0.9656464709556527, 'recall': 0.9572755417956657, 'f1': 0.9614427860696517, 'number': 1615}, 'eval_MISC': {'precision': 0.8542141230068337, 'recall': 0.8751458576429405, 'f1': 0.8645533141210375, 'number': 857}, 'eval_ORG': {'precision': 0.9405981155264236, 'recall': 0.9526970954356846, 'f1': 0.9466089466089466, 'number': 2410}, 'eval_PER': {'precision': 0.9705014749262537, 'recall': 0.9815035799522673, 'f1': 0.9759715218036191, 'number': 1676}, 'eval_overall_precision': 0.9428571428571428, 'eval_overall_recall': 0.9510521500457456, 'eval_overall_f1': 0.9469369164199499, 'eval_overall_accuracy': 0.9903980413534553, 'eval_f1_micro': 0.9903980413534553, 'eval_f1_macro': 0.9473213166976306, 'eval_f1_weighted': 0.9903744973929919, 'eval_runtime': 10.7299, 'eval_samples_per_second': 384.252, 'eval_steps_per_second': 24.045, 'epoch': 5.0}




0

: 